# 02 · Reproductive Feature Engineering
## NHANES 2017–March 2020 Women's CKM Phenotyping Project

---

**Author:** Alexandra Velez, OB-GYN (Colombia) | Data Science  
**Input:** `data/processed/nhanes_clean_sample.csv` — 1,603 women aged 20–44  
**Output:** `data/processed/reproductive_features.csv`  
**Last updated:** 2026

---

### Purpose of this notebook

This notebook engineers reproductive health features across four clinical 
domains — menstrual history, pregnancy history and adverse outcomes, surgical 
history, and hormone therapy — from the cleaned analytical sample produced in 
notebook 01.

The central clinical question driving feature construction is:

> Among reproductive-age women aged 20–44, do those with adverse pregnancy 
> outcomes already show distinct early cardiometabolic risk profiles that 
> are identifiable through unsupervised clustering?

This question was deliberately scoped to the 20–44 age range, driven by 
both a data limitation and a clinical opportunity. CDC suppresses pregnancy and hysterectomy data for women over 44 in 
the public use file due to disclosure risk, restricting the analytical sample 
to reproductive-age women only. However, this constraint aligns with a genuine 
clinical opportunity: women aged 20–44 represent the intervention window before 
overt cardiovascular disease manifests. Their cardiometabolic biomarkers are 
already measurable and may already be diverging based on reproductive history — 
even if cardiovascular outcomes have not yet had time to develop. The analysis 
therefore focuses on early cardiometabolic dysregulation rather than late-stage 
outcomes, which is both the honest scope of what this data supports and the 
clinically meaningful question for preventive medicine.

Features engineered here will be used to identify early cardiometabolic risk 
profiles in notebook 05. This analysis makes no causal claims — it identifies 
associations between reproductive history and concurrent metabolic biomarker 
profiles. Longitudinal data with outcome follow-up would be required to 
establish causal relationships between adverse pregnancy outcomes and 
cardiovascular disease.

---

### Notebook inputs and outputs

| File | Description |
|---|---|
| `data/processed/nhanes_clean_sample.csv` | Analytical sample from notebook 01 |
| `data/processed/reproductive_features.csv` | Engineered feature matrix — input for notebook 03 |

---

### Engineering decisions carried forward from notebook 01

The following decisions made during exploration directly affect feature 
construction in this notebook:

| Variable | Issue | Action |
|---|---|---|
| RHD018 | Age in months, not years | Divide by 12 |
| RHQ010 | Code 0 = pre-menarche | Handle explicitly — not missing |
| RHQ160 | Code 11 = 11 or more pregnancies | Top-coded — treat as 11 |
| RHD167 | Code 5 = 5 or more deliveries | Top-coded — treat as 5 |
| RHQ162 | Code 3 = borderline GDM | Treat as Yes — conservative clinical decision |
| RHQ020 | Ordinal fallback for menarche age | Use midpoint imputation |
| RHQ070 | Ordinal fallback for menopause age | Use midpoint imputation |
| RHQ542A–D | Multi-select hormone therapy forms | Build composite HRT_Type |

## Section 1 · Imports & Setup

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from pathlib import Path
import warnings
import os

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_PATH  = Path('../data/processed/nhanes_clean_sample.csv')
OUTPUT_PATH = Path('../data/processed/reproductive_features.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')
print(f'  Input:  {INPUT_PATH}')
print(f'  Output: {OUTPUT_PATH}')

Setup complete.
  Input:  ../data/processed/nhanes_clean_sample.csv
  Output: ../data/processed/reproductive_features.csv


---
## Section 2 · Load Analytical Sample

The analytical sample was fully validated and documented in notebook 01. 
The three assertions below confirm the CSV handoff was clean — shape, 
index, and age range.

In [13]:
# ── Load analytical sample ─────────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH, index_col='SEQN')

# Light validation — confirms handoff from notebook 01 was clean
assert df.shape == (1603, 42), \
    f'Unexpected shape {df.shape} — expected (1603, 42)'
assert df.index.name == 'SEQN', \
    'SEQN is not the index — check CSV export from notebook 01'
assert df['RIDAGEYR'].between(20, 44).all(), \
    'Age restriction violated — participants outside 20–44 present'

print(f'✓ Analytical sample loaded: {df.shape}')
print(f'  Age range:  {df["RIDAGEYR"].min():.0f}–{df["RIDAGEYR"].max():.0f} years')
print(f'  SEQN index: confirmed')
print(f'  Columns:    {df.shape[1]} '
      f'(30 P_RHQ variables + 12 P_DEMO variables)')

✓ Analytical sample loaded: (1603, 42)
  Age range:  20–44 years
  SEQN index: confirmed
  Columns:    42 (30 P_RHQ variables + 12 P_DEMO variables)


---
## Section 3 · Binary Variable Recoding

All binary variables in the analytical sample retain their raw NHANES coding 
(1=Yes, 2=No) from notebook 01 — recoding was deliberately deferred to this 
notebook to preserve the boundary between exploration and engineering.

This section applies permanent recoding across all binary variables using 
`recode_yes_no()`, converting 1→1.0 and 2→0.0 with NaN preserved. One 
variable requires special handling before recoding:

**RHQ162 — Gestational diabetes:**
Code 3 (borderline GDM) is treated as Yes (1.0) — a conservative clinical 
decision. Borderline GDM carries similar metabolic implications to confirmed 
GDM and represents the same underlying pathophysiology of impaired glucose 
tolerance under metabolic stress. Collapsing it into the Yes category is 
the clinically appropriate choice for a cardiometabolic risk analysis.


In [14]:
# ── Binary variable recoding ───────────────────────────────────────────────
# Permanently recode all binary variables from NHANES coding (1=Yes, 2=No)
# to analytical coding (1.0=Yes, 0.0=No, NaN=missing)
#
# This is a permanent transformation — df is modified in place
# All downstream feature engineering uses the recoded values

def recode_yes_no(series):
    """Convert NHANES binary coding (1=Yes, 2=No) to (1.0=Yes, 0.0=No, NaN)."""
    return series.map({1.0: 1.0, 2.0: 0.0})

# Binary variables in P_RHQ
binary_vars = [
    'RHQ031',   # Regular periods in past 12 months
    'RHQ074',   # Tried ≥1 year to conceive without success
    'RHQ076',   # Seen doctor for inability to conceive
    'RHQ078',   # Ever treated for PID
    'RHQ131',   # Ever been pregnant
    'RHD143',   # Currently pregnant
    'RHQ172',   # Any baby weighed 9 lbs or more
    'RHQ200',   # Currently breastfeeding
    'RHD280',   # Had a hysterectomy
    'RHQ305',   # Had both ovaries removed
    'RHQ540',   # Ever used female hormones
    'RHQ554',   # Ever used estrogen-only pills
    'RHQ570',   # Ever used estrogen/progestin combo pills
]

# ── Special handling: RHQ162 borderline GDM → Yes ─────────────────────────
# Code 3 (borderline GDM) treated as Yes before recoding
# Clinical rationale: borderline GDM reflects same underlying impaired
# glucose tolerance as confirmed GDM — conservative classification
n_borderline = (df['RHQ162'] == 3).sum()
df['RHQ162'] = df['RHQ162'].replace(3, 1)
print(f'RHQ162 borderline GDM (code 3) → Yes: {n_borderline:,} participants reclassified')

# Add RHQ162 to binary vars after special handling
binary_vars.append('RHQ162')

# ── Apply recoding ─────────────────────────────────────────────────────────
print(f'\nApplying recode_yes_no() to {len(binary_vars)} binary variables...')
for var in binary_vars:
    df[var] = recode_yes_no(df[var])

# ── Validate ───────────────────────────────────────────────────────────────
print('\nValidation — no raw NHANES codes (1.0/2.0) should remain:')
issues = []
for var in binary_vars:
    if (df[var] == 2.0).any():
        issues.append(var)

if issues:
    print(f'  ⚠️  Raw coding still present in: {issues}')
else:
    print(f'  ✓ All {len(binary_vars)} variables confirmed recoded')
    print(f'  ✓ Only 1.0, 0.0, and NaN present in all binary variables')

# ── Summary ────────────────────────────────────────────────────────────────
print(f'\nBinary variable recoding summary:')
print(f'  {"Variable":<12} {"Yes (1.0)":>10} {"No (0.0)":>10} {"NaN":>8} {"% Yes":>8}')
print(f'  {"─"*12} {"─"*10} {"─"*10} {"─"*8} {"─"*8}')
for var in binary_vars:
    n_yes  = (df[var] == 1.0).sum()
    n_no   = (df[var] == 0.0).sum()
    n_nan  = df[var].isna().sum()
    pct    = n_yes / (n_yes + n_no) * 100 if (n_yes + n_no) > 0 else 0
    print(f'  {var:<12} {n_yes:>10,} {n_no:>10,} {n_nan:>8,} {pct:>7.1f}%')

RHQ162 borderline GDM (code 3) → Yes: 6 participants reclassified

Applying recode_yes_no() to 14 binary variables...

Validation — no raw NHANES codes (1.0/2.0) should remain:
  ✓ All 14 variables confirmed recoded
  ✓ Only 1.0, 0.0, and NaN present in all binary variables

Binary variable recoding summary:
  Variable      Yes (1.0)   No (0.0)      NaN    % Yes
  ──────────── ────────── ────────── ──────── ────────
  RHQ031            1,435        166        2    89.6%
  RHQ074              191      1,409        3    11.9%
  RHQ076              123      1,478        2     7.7%
  RHQ078               90      1,500       13     5.7%
  RHQ131            1,150        451        2    71.8%
  RHD143               67      1,009      527     6.2%
  RHQ172              144        917      542    13.6%
  RHQ200               58        177    1,368    24.7%
  RHD280               61      1,523       19     3.9%
  RHQ305               15      1,565       23     0.9%
  RHQ540               66     

In [15]:
# Verify hormone therapy routing
n_hrt_users = (df['RHQ540'] == 1.0).sum()
n_estrogen_only = (df['RHQ554'] == 1.0).sum()
n_combo = (df['RHQ570'] == 1.0).sum()

print(f'Ever used hormones (RHQ540=Yes):     {n_hrt_users:,}')
print(f'Among hormone users:')
print(f'  Estrogen-only pills (RHQ554=Yes):  {n_estrogen_only:,} '
      f'({n_estrogen_only/n_hrt_users*100:.1f}% of users)')
print(f'  Combo pills (RHQ570=Yes):          {n_combo:,} '
      f'({n_combo/n_hrt_users*100:.1f}% of users)')
print(f'\nAs % of full analytical sample (n=1,603):')
print(f'  Estrogen-only pills:               '
      f'{n_estrogen_only/len(df)*100:.1f}%')
print(f'  Combo pills:                       '
      f'{n_combo/len(df)*100:.1f}%')

Ever used hormones (RHQ540=Yes):     66
Among hormone users:
  Estrogen-only pills (RHQ554=Yes):  11 (16.7% of users)
  Combo pills (RHQ570=Yes):          6 (9.1% of users)

As % of full analytical sample (n=1,603):
  Estrogen-only pills:               0.7%
  Combo pills:                       0.4%


### Recoding complete — key observations

All 14 binary variables successfully recoded from NHANES coding (1=Yes, 2=No) 
to analytical coding (1.0=Yes, 0.0=No, NaN=missing). Six women with borderline 
GDM (RHQ162 code 3) were reclassified as Yes before recoding — a conservative 
clinical decision reflecting that borderline GDM represents the same underlying 
impaired glucose tolerance as confirmed GDM.

Several patterns in the recoding summary are worth noting before feature 
engineering begins:

**Structural missingness from skip logic:**
- RHD143 (currently pregnant): 527 NaN — only administered to women who 
  answered Yes to RHQ131 (ever pregnant). Women who have never been pregnant 
  were never asked this question — their NaN is structural, not missing data.
- RHQ172 (macrosomia): 542 NaN — same gate as RHD143
- RHQ200 (breastfeeding): 1,368 NaN — asked only of women who delivered 
  in the past 2 years, extremely sparse

**Hormone therapy variables:**
- RHQ540 (ever used hormones): only 66 Yes (4.1%) — expected for 
  reproductive-age women 20–44
- RHQ554 (Use hormone pills w/estrogen only) and RHQ570 (Used estrogen/progestin combo pills): sparse by design, gated behind RHQ540
- These will be combined into a single composite feature in Section 7

**Key exposure variables:**
- RHQ131 (ever pregnant): 71.8% Yes — primary stratification variable
- RHQ162 (GDM): 11.8% Yes among those asked — primary APO exposure
- RHQ172 (macrosomia): 13.6% Yes among those asked

---
## Section 4 · Domain 1: Menstrual History Features

Menstrual history variables capture the reproductive lifespan — from menarche 
to menopause — and provide context for understanding estrogen exposure duration, 
cycle regularity, and reproductive aging. These features are clinically relevant 
to CKM risk because estrogen has protective effects on cardiovascular and 
metabolic function. Earlier menopause, shorter reproductive span, and irregular 
cycles are all associated with adverse cardiometabolic trajectories.

Four features are derived in this section:

- **Age_Menarche** — age at first menstrual period in years, derived primarily 
  from RHQ010 (exact age in years, asked of all females aged 12–150). For women 
  who refused or did not know their exact age, ordinal midpoint imputation is 
  applied using RHQ020 (age range at first menstrual period), which provides a 
  range category rather than an exact age. The midpoint of each range is assigned 
  as the best available estimate.

- **Irregular_Periods** — binary flag for absence of regular periods in the 
  past 12 months, derived from RHQ031. A value of 1.0 indicates irregular or 
  absent periods — the clinically meaningful direction for CKM risk.

- **Menopause_Status** — categorical variable distinguishing premenopausal, 
  surgically menopausal, and naturally menopausal women. Derived from the 
  combination of RHQ031 (regular periods), RHD043 (reason for no periods), 
  RHD280 (hysterectomy), and RHQ305 (oophorectomy).

- **Age_Menopause** — age at menopause in years, derived from RHQ060 (exact 
  age) with ordinal midpoint imputation from RHQ070 where needed. Calculated 
  only for postmenopausal women.

- **Reproductive_Span** — continuous variable capturing total years of 
  endogenous estrogen exposure (Age_Menopause − Age_Menarche). Calculated 
  only for postmenopausal women with both ages known. Women aged 20–44 who 
  report natural menopause represent premature ovarian insufficiency — a 
  clinically recognized condition associated with accelerated cardiometabolic 
  risk.

### Engineering decisions for this domain

**RHQ010 = 0 (pre-menarche):**
One woman in the analytical sample has RHQ010 = 0, indicating menarche had 
not yet started at time of interview at age 27. This is consistent with 
primary amenorrhea. This woman will have NaN for all menstrual features and 
contributes to non-menstrual analyses only.

**RHQ020 ordinal midpoint values:**

| Code | Range | Midpoint assigned |
|---|---|---|
| 1 | Younger than 9 years | 8.5 |
| 2 | 9–10 years | 9.5 |
| 3 | 11–12 years | 11.5 |
| 4 | 13–14 years | 13.5 |
| 5 | 15–16 years | 15.5 |
| 6 | 17–19 years | 18.0 |
| 7 | 20 years or older | 20.0 |

**Menopause_Status derivation logic:**
Menopause status cannot be read from a single variable — it must be inferred 
from multiple variables. The derivation priority is:

1. Surgical menopause — hysterectomy (RHD280=1) OR bilateral oophorectomy 
   (RHQ305=1) → Surgical_Menopause
2. Natural menopause — no regular periods (RHQ031=0) AND reason is menopause 
   (RHD043=2) AND no surgical history → Natural_Menopause
3. All others with valid menstrual data → Premenopausal

In [24]:
# ── Domain 1: Menstrual history features ──────────────────────────────────
# Features derived in this section:
#   Age_Menarche      — age at first menstrual period in years
#   Irregular_Periods — binary flag for unexplained menstrual irregularity
#   Menopause_Status  — premenopausal vs surgical menopause
#   Age_Menopause     — age at surgical menopause (surgical only)
#   Reproductive_Span — years of endogenous estrogen exposure (surgical only)

features = pd.DataFrame(index=df.index)

# ── Feature 1: Age_Menarche ────────────────────────────────────────────────
# Primary source: RHQ010 (exact age in years, asked 12–150)
# Fallback: RHQ020 ordinal midpoint imputation where RHQ010 is NaN
# Exclusion: RHQ010 == 0 (primary amenorrhea) → NaN

ordinal_midpoints_menarche = {
    1.0:  8.5,   # < 9 years
    2.0:  9.5,   # 9–10 years
    3.0: 11.5,   # 11–12 years
    4.0: 13.5,   # 13–14 years
    5.0: 15.5,   # 15–16 years
    6.0: 18.0,   # 17–19 years
    7.0: 20.0,   # ≥ 20 years
}

age_menarche     = df['RHQ010'].copy().astype(float)
premenarche_mask = age_menarche == 0.0
age_menarche[premenarche_mask] = np.nan

rhq020_imputed = df['RHQ020'].map(ordinal_midpoints_menarche)
n_imputed      = (age_menarche.isna() & rhq020_imputed.notna()).sum()
age_menarche   = age_menarche.fillna(rhq020_imputed)

features['Age_Menarche'] = age_menarche

print('Age_Menarche construction:')
print(f'  From RHQ010 (exact age):         '
      f'{(df["RHQ010"].notna() & (df["RHQ010"] > 0)).sum():,}')
print(f'  From RHQ020 (ordinal imputed):   {n_imputed:,}')
print(f'  Pre-menarche (RHQ010=0) → NaN:  {premenarche_mask.sum():,}')
print(f'  Total valid:                     '
      f'{features["Age_Menarche"].notna().sum():,}')
print(f'  Total NaN:                       '
      f'{features["Age_Menarche"].isna().sum():,}')

# ── Feature 2: Irregular_Periods ──────────────────────────────────────────
# RHQ031 = 0 (no regular periods in past 12 months)
# Refined using RHD043 (reason for no periods) to exclude known
# non-CKM-relevant causes:
#   Code 1 = Pregnant       → exclude
#   Code 2 = Breastfeeding  → exclude
#   Code 3 = Hysterectomy   → exclude (captured in Menopause_Status)
#   Code 7 = Menopause      → absent in 20–44 sample
#   NaN    = Unknown reason → retained conservatively
#
# Final flag captures unexplained menstrual irregularity only

irregular_base    = df['RHQ031'] == 0.0
exclude_irregular = (
    (df['RHD043'] == 1.0) |   # pregnant
    (df['RHD043'] == 2.0) |   # breastfeeding
    (df['RHD043'] == 3.0)     # hysterectomy
)

features['Irregular_Periods'] = np.where(
    df['RHQ031'].isna(), np.nan,
    np.where(irregular_base & ~exclude_irregular, 1.0, 0.0)
)

print(f'\nIrregular_Periods construction:')
print(f'  Raw No to RHQ031:              {irregular_base.sum():,}')
print(f'  Excluded — pregnant:           '
      f'{(irregular_base & (df["RHD043"] == 1.0)).sum():,}')
print(f'  Excluded — breastfeeding:      '
      f'{(irregular_base & (df["RHD043"] == 2.0)).sum():,}')
print(f'  Excluded — hysterectomy:       '
      f'{(irregular_base & (df["RHD043"] == 3.0)).sum():,}')
print(f'  Retained — unknown reason:     '
      f'{(irregular_base & df["RHD043"].isna()).sum():,}')
print(f'  Final Irregular_Periods = 1:   '
      f'{(features["Irregular_Periods"] == 1.0).sum():,}')

# ── Feature 3: Menopause_Status ───────────────────────────────────────────
# Two categories only — Natural_Menopause not identifiable in 20–44 sample
# RHD043 code 7 (menopause) is absent in this age group
#
# Surgical_Menopause: hysterectomy (RHD280=1) OR oophorectomy (RHQ305=1)
# Premenopausal: all others with valid menstrual data

menopause_status              = pd.Series('Premenopausal', index=df.index,
                                          dtype=object)
surgical_mask                 = (df['RHD280'] == 1.0) | (df['RHQ305'] == 1.0)
menopause_status[surgical_mask]    = 'Surgical_Menopause'
menopause_status[premenarche_mask] = np.nan

features['Menopause_Status'] = menopause_status

print(f'\nMenopause_Status construction:')
for status in ['Premenopausal', 'Surgical_Menopause']:
    n   = (features['Menopause_Status'] == status).sum()
    pct = n / len(features) * 100
    print(f'  {status:<22} {n:>5,}  ({pct:.1f}%)')
print(f'  {"NaN":<22} '
      f'{features["Menopause_Status"].isna().sum():>5,}')
print(f'\n  Note: Natural_Menopause not derivable — RHD043 code 7')
print(f'  (menopause) absent in women aged 20–44')

# ── Features 4 & 5: Age_Menopause and Reproductive_Span ───────────────────
# Restricted to Surgical_Menopause women only
# Age_Menopause: RHQ060 (exact age) with RHQ070 ordinal fallback
# Reproductive_Span: Age_Menopause - Age_Menarche

ordinal_midpoints_menopause = {
    1.0: 20.0,   # < 25 years
    2.0: 27.0,   # 25–29 years
    3.0: 32.0,   # 30–34 years
    4.0: 37.0,   # 35–39 years
    5.0: 42.0,   # 40–44 years
    6.0: 47.0,   # 45–49 years
    7.0: 52.0,   # 50–54 years
    8.0: 55.0,   # ≥ 55 years
}

# Postmenopausal mask — surgical only after correction
postmeno_mask  = features['Menopause_Status'] == 'Surgical_Menopause'

age_menopause  = df['RHQ060'].copy().astype(float)
rhq070_imputed = df['RHQ070'].map(ordinal_midpoints_menopause)
age_menopause  = age_menopause.fillna(rhq070_imputed)

# Restrict to surgical menopause only
features['Age_Menopause'] = np.nan
features.loc[postmeno_mask, 'Age_Menopause'] = age_menopause[postmeno_mask]

# Reproductive span
features['Reproductive_Span'] = np.nan
features.loc[postmeno_mask, 'Reproductive_Span'] = (
    features.loc[postmeno_mask, 'Age_Menopause'] -
    features.loc[postmeno_mask, 'Age_Menarche']
)

# ── Domain 1 summary ───────────────────────────────────────────────────────
print(f'\n{"═"*60}')
print(f'DOMAIN 1 SUMMARY — Menstrual History Features')
print(f'{"═"*60}\n')

print(f'  {"Feature":<22} {"Valid":>6} {"NaN":>6}  {"Key statistic"}')
print(f'  {"─"*22} {"─"*6} {"─"*6}  {"─"*30}')

# Age_Menarche
v = features['Age_Menarche'].notna().sum()
print(f'  {"Age_Menarche":<22} {v:>6,} '
      f'{len(features)-v:>6,}  '
      f'Mean {features["Age_Menarche"].mean():.1f} yrs  '
      f'Range {features["Age_Menarche"].min():.0f}–'
      f'{features["Age_Menarche"].max():.0f}')

# Irregular_Periods
v = (features['Irregular_Periods'] == 1.0).sum()
print(f'  {"Irregular_Periods":<22} {v:>6,} '
      f'{features["Irregular_Periods"].isna().sum():>6,}  '
      f'{v/len(features)*100:.1f}% of analytical sample')

# Menopause_Status
v = features['Menopause_Status'].notna().sum()
s = (features['Menopause_Status'] == 'Surgical_Menopause').sum()
print(f'  {"Menopause_Status":<22} {v:>6,} '
      f'{features["Menopause_Status"].isna().sum():>6,}  '
      f'{s} surgical menopause ({s/len(features)*100:.1f}%)')

# Age_Menopause
v = features['Age_Menopause'].notna().sum()
print(f'  {"Age_Menopause":<22} {v:>6,} '
      f'{len(features)-v:>6,}  '
      f'Mean {features["Age_Menopause"].mean():.1f} yrs  '
      f'(surgical only)')

# Reproductive_Span
v = features['Reproductive_Span'].notna().sum()
print(f'  {"Reproductive_Span":<22} {v:>6,} '
      f'{len(features)-v:>6,}  '
      f'Mean {features["Reproductive_Span"].mean():.1f} yrs  '
      f'(surgical only)')

# Validation
assert features['Age_Menopause'].notna().sum() <= \
    (features['Menopause_Status'] == 'Surgical_Menopause').sum(), \
    'Age_Menopause assigned to non-surgical women'
assert (features.loc[
    features['Menopause_Status'] == 'Premenopausal', 
    'Age_Menopause']).isna().all(), \
    'Premenopausal women have Age_Menopause values'

print(f'\n✓ All Domain 1 assertions passed')

Age_Menarche construction:
  From RHQ010 (exact age):         1,519
  From RHQ020 (ordinal imputed):   6
  Pre-menarche (RHQ010=0) → NaN:  1
  Total valid:                     1,525
  Total NaN:                       78

Irregular_Periods construction:
  Raw No to RHQ031:              166
  Excluded — pregnant:           17
  Excluded — breastfeeding:      5
  Excluded — hysterectomy:       53
  Retained — unknown reason:     91
  Final Irregular_Periods = 1:   91

Menopause_Status construction:
  Premenopausal          1,539  (96.0%)
  Surgical_Menopause        63  (3.9%)
  NaN                        1

  Note: Natural_Menopause not derivable — RHD043 code 7
  (menopause) absent in women aged 20–44

════════════════════════════════════════════════════════════
DOMAIN 1 SUMMARY — Menstrual History Features
════════════════════════════════════════════════════════════

  Feature                 Valid    NaN  Key statistic
  ────────────────────── ────── ──────  ───────────────────────────

### Domain 1 results — menstrual history features

All five menstrual history features were derived successfully. Several 
engineering decisions required careful use of the codebook — particularly 
for Irregular_Periods and Menopause_Status, where raw variable values alone 
were insufficient.

**Age_Menarche** — 1,525 of 1,603 women (95.1%) have a valid menarche age. 
The mean of 12.6 years is consistent with published US menarche age 
distributions. Six women were recovered via RHQ020 ordinal midpoint 
imputation — a negligible contribution as anticipated from the coverage 
analysis in notebook 01. One woman with RHQ010 = 0 (primary amenorrhea, 
age 27) has NaN for all menstrual features.

**Irregular_Periods — retained for descriptive purposes only**

This feature captures 91 women (5.7% of the analytical sample) with 
unexplained menstrual irregularity after excluding women with known 
non-CKM explanations (pregnant, breastfeeding, hysterectomy). The 
remaining cases likely represent a heterogeneous mix of causes — 
hormonal contraception, PCOS, hypothalamic amenorrhea, thyroid dysfunction, 
hyperprolactinemia, and other medications affecting the hypothalamic-
pituitary-ovarian axis — that cannot be fully distinguished from available 
data. Complete characterization of menstrual irregularity cause would require clinical 
data beyond what any population survey can capture.

At 5.7% prevalence with an unresolvable heterogeneous confound, this 
feature has insufficient signal for clustering. It is retained in the 
feature matrix for descriptive subgroup analysis in notebook 04 but 
explicitly excluded from the clustering feature set in notebook 05.

**Menopause_Status** — restricted to two categories after discovering that 
RHD043 code 7 (menopause/change of life) is entirely absent in women aged 
20–44. Natural menopause cannot be identified in this analytical sample. 
The 63 women with surgical menopause (3.9%) — defined by hysterectomy 
(RHD280=1) or bilateral oophorectomy (RHQ305=1) — represent a clinically 
distinct group with abrupt estrogen loss, a known risk factor for accelerated 
cardiometabolic dysfunction.

**Age_Menopause and Reproductive_Span** — restricted to the 63 surgically 
menopausal women. Age_Menopause is valid for 62 of 63 (one missing). The 
mean surgical menopause age of 33.4 years reflects the relatively young age 
at which these women underwent surgery. Reproductive_Span is valid for 58 
women — mean 21.2 years, substantially shorter than the population average 
of approximately 37 years — consistent with premature surgical estrogen loss.

**Note for clustering:** Age_Menopause and Reproductive_Span will be sparse 
features in the clustering analysis — calculable for only 58–62 women (3.6–
3.9% of the analytical sample). These features are retained for descriptive 
purposes and subgroup analysis but are unlikely to drive cluster separation 
given their sparsity.